In [1]:
from torch import nn
import torch


In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weights = nn.Parameter(torch.ones(dim))
    def forward(self, X):
        var_X = x.pow(2).mean(dim=1, keepdim=True)
        x_normed = X * torch.sqrt(var_X + self.eps)
        return x_normed * self.weights

In [3]:
import torch.nn.functional as F

class SwiGLU(nn.Module):
    def __init__(self, d_model, ffn_dim):
        super().__init__()
        self.w_gate = nn.Linear(d_model, ffn_dim, bias=False)
        self.w_up = nn.Linear(d_model, ffn_dim, bias=False)
        self.w_down = nn.Linear(ffn_dim, d_model, bias=False)
    def forward(self, X):
        gate = F.silu(self.w_gate(X))
        up = self.w_up(X)
        down = self.w_down(gate * up)
        return down
        

In [17]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_heads
        self.num_queries_per_kv = num_heads // num_kv_heads # Коэффициент повтора GQA (g)

        self.q_proj = nn.Linear(d_model, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(num_heads * self.head_dim, d_model, bias=False)
        
    def forward(self, X, kv_cache=None):
        # X -> [B, T_q, D]
        B, T_q, D = X.shape
        q = self.q_proj(X).view(B, T_q, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T_q, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T_q, self.num_kv_heads, self.head_dim).transpose(1, 2)

        if kv_cache is not None:
            k_prev, v_prev = kv_cache
            k = torch.cat([k_prev, k_new], dim=2)
            v = torhc.cat([v_prev, v_new], dim=2)
        else:
            k, v = k_new, v_new
        new_kv_cache = (k, v)
        if self.num_queries_per_kv > 1:
            # [B, H_kv, T_total, d_h] -> [B, H, T_total, d_h]
            k_exec = torch.repeat_interleave(k, dim=1, repeats=self.num_queries_per_kv)
            v_exec = torch.repeat_interleave(v, dim=1, repeats=self.num_queries_per_kv)
        else:
            k_exec, v_exec = k, v

        scores = torch.matmul(q, k_exec.transpose(-1, -2)) // math.sqrt(self.head_dim)
        T_total = k.size(2)
        T_past = T_total - T_q
        if T_q > 1:
            causal_mask = torch.triu(torch.full((T_q, T_total), float('-inf'), device=X.device),
                diagonal=T_past + 1)
            scores = scores + causal_mask.unsqueeze(0).unsqueeze(0)
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, v_exec).transpose(1, 2).contiguous().view(B, T_q, -1)
        out = self.out_proj(context)
        return out, new_kv_cache

In [5]:
d_model = 200
num_kv_heads = 4
head_dim = 2
a = nn.Linear(d_model, num_kv_heads * head_dim, bias=False)

In [6]:
x = torch.randn(200)

In [7]:
x_ = a(x)

In [8]:
x_

tensor([-0.1602, -0.6543,  1.2769,  0.1716, -0.1033,  0.3460, -0.5148, -0.5804],
       grad_fn=<SqueezeBackward4>)

In [55]:
x = torch.randn(16, 300, 12, 4)
print(x.shape)
x = x.transpose(1, 2)
print(x.shape)

torch.Size([16, 300, 12, 4])
torch.Size([16, 12, 300, 4])


In [15]:
a = torch.randn(2,2)
b = torch.randn(2,2)
v = torch.cat([a, b], dim=1)
v

tensor([[-1.9162, -1.3990,  0.5622, -1.0408],
        [-1.1077,  1.1444,  0.5827,  1.5253]])

In [46]:
x

tensor([[[[ 7.8906e-01,  5.9601e-01, -1.7631e+00,  1.9455e+00],
          [-1.4667e-01, -2.5400e-01, -4.5314e-01, -1.3908e+00],
          [-9.1329e-01, -5.8337e-02,  5.5808e-01, -2.8251e-02],
          ...,
          [ 1.9766e-01, -1.4241e+00, -3.0186e-01,  2.4302e-01],
          [-9.9752e-01,  5.7399e-01, -1.6701e+00, -1.0311e+00],
          [-1.4669e+00,  1.3090e+00,  6.9568e-01, -2.2919e-01]],

         [[-7.5242e-01, -4.2539e-01, -6.7560e-01, -9.1377e-02],
          [-3.6613e-01,  1.9568e-02,  1.6346e+00,  5.6632e-01],
          [ 1.8802e+00,  3.9106e-01,  5.0685e-01,  1.0662e+00],
          ...,
          [ 4.8726e-01, -9.2273e-01, -1.9269e+00,  1.0091e-01],
          [ 3.7363e-01,  4.2460e-01,  3.5446e-01, -4.5339e-01],
          [-1.3832e-01, -3.2210e+00, -1.4445e+00,  4.2832e-01]],

         [[-9.3227e-01,  6.9395e-02, -1.2472e+00, -2.5331e-01],
          [ 2.8185e-01, -1.2791e+00, -6.2401e-01, -5.7438e-01],
          [ 7.8011e-01, -1.2357e+00,  1.7478e+00, -7.1285e-01],
      